In [2]:
import zenidatasdk as zd
import alphalens
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import datetime
import time
import os

pd.set_option('expand_frame_repr', False)
pd.set_option('display.max_rows', 1000)
pd.set_option('display.max_colwidth', 100)

In [3]:
# 检查版本
print(f"ZeniData SDK版本: {zd.__version__}")

# 创建客户端（测试基础功能）
client = zd.Client()
print("✅ 客户端创建成功")

# 登录账号，初始化 client 对象
client = zd.Client(
    username = "3191883175@qq.com",
    password = "Abc123456",
    base_url = "http://192.168.1.100:8000")
print("✅ 登录成功")

ZeniData SDK版本: 2.0.5
✅ 客户端创建成功
✅ 登录成功


In [30]:
start_date =  "2016-01-01"                          # 定义所要回测的起始日期
current_day = str(datetime.datetime.now().date())   # 使用现在作为结束日期
get_index_constituents_df = (                       # 定义快捷的指数成分股表格获取
    lambda index: client.get_index_constituents_df(index, start_date = start_date, end_date = current_day)
)

core_index_constituents_dict = {
    index_code: get_index_constituents_df(index_code)
    for index_code in [
        rf"000300.XSHG",  # 沪深300
        rf"000905.XSHG",  # 中证500
        rf"000852.XSHG",  # 中证1000
        rf"000016.XSHG",  # 上证50
        rf"399006.XSHE",  # 创业板
    ]
}

HS300 = core_index_constituents_dict[rf"000300.XSHG"]  # 沪深300
ZZ500 = core_index_constituents_dict[rf"000905.XSHG"]  # 中证500
ZZ1000 = core_index_constituents_dict[rf"000852.XSHG"]  # 中证1000
SZ50 = core_index_constituents_dict[rf"000016.XSHG"]  # 上证50
CYB = core_index_constituents_dict[rf"399006.XSHE"]  # 创业板

symbols_df = client.get_symbols_df()  # 股票全集
symbols_df

,symbol,name,short_name,start_date,end_date,update_time
0,000001.XSHE,平安银行,PAYH,1991-04-03,2200-01-01,2026-04-13 15:15:00.989
1,000002.XSHE,万科A,WKA,1991-01-29,2200-01-01,2026-04-13 15:15:00.989
2,000004.XSHE,*ST国华,*STGH,1990-12-01,2200-01-01,2026-04-13 15:15:00.989
3,000005.XSHE,ST星源,STXY,1990-12-10,2024-04-25,2026-04-13 15:15:00.989
4,000006.XSHE,深振业A,SZYA,1992-04-27,2200-01-01,2026-04-13 15:15:00.989
...,...,...,...,...,...,...
5489,688816.XSHG,易思维,YSW,2026-02-11,2200-01-01,2026-04-13 15:15:00.989
5490,688818.XSHG,电科蓝天,DKLT,2026-02-10,2200-01-01,2026-04-13 15:15:00.989
5491,688819.XSHG,天能股份,TNGF,2021-01-18,2200-01-01,2026-04-13 15:15:00.989
5492,688981.XSHG,中芯国际,ZXGJ,2020-07-16,2200-01-01,2026-04-13 15:15:00.989


In [ ]:
bars_list = []
for index, item in symbols_df.iterrows():  # 遍历指定截止日期的 股票列表DataFrame
    symbol = item['symbol']  # 赋值股票代码等变量
    name = item['name']
    short_name = item['short_name']
    start_date = item['start_date']
    end_date = item['end_date']
    # print(f"{index} {symbol} {name} {short_name}")

    bars_df = client.get_bars(  # 则获取股票 OHLCV 数据
        symbol = symbol,
        start_date = start_date,
        end_date = current_day,
        frequency = '1d',
        adjust_type = 'post',
        market = 'cn_stock'
    )
    bars_list.append(bars_df)


# bars_df.info()  # 查看最后一个股票的结构
# print(bars_df.head())  # 打印 DataFrame 的首尾
# print(bars_df.tail())  # 用于 copy/paste 方便询问 ai